In [1]:
import json
from collections import Counter

# ── Paths ─────────────────────────────────────────────────────────────────────
INPUT_PATH  = "cjpe_train_rr_segmented_CPU.jsonl"
OUTPUT_PATH = "cjpe_track_issue_analysis_extracted.jsonl"

# ── Counters ──────────────────────────────────────────────────────────────────
extracted       = 0
empty_issue     = 0
empty_analysis  = 0

# ── Extraction ────────────────────────────────────────────────────────────────
with open(INPUT_PATH, "r", encoding="utf-8") as fin, \
     open(OUTPUT_PATH, "w", encoding="utf-8") as fout:

    for line in fin:
        line = line.strip()
        if not line:
            continue

        rec = json.loads(line)

        out = {
            "id"       : rec.get("id", ""),
            "ISSUE"    : rec.get("ISSUE",    "").strip(),
            "ANALYSIS" : rec.get("ANALYSIS", "").strip(),
            "label"    : rec.get("label", -1),
        }

        # ── Track empty fields ────────────────────────────────────────────────
        if not out["ISSUE"]:    empty_issue    += 1
        if not out["ANALYSIS"]: empty_analysis += 1

        fout.write(json.dumps(out, ensure_ascii=False) + "\n")
        extracted += 1

# ═════════════════════════════════════════════════════════════════════════════
# Summary
# ═════════════════════════════════════════════════════════════════════════════
print("=" * 55)
print("  ISSUE + ANALYSIS EXTRACTION COMPLETE")
print("=" * 55)
print(f"  Input          : {INPUT_PATH}")
print(f"  Output         : {OUTPUT_PATH}")
print(f"  Total records  : {extracted:,}")
print()
print(f"  Field coverage (docs that HAVE the field):")
print(f"    ISSUE    : {extracted - empty_issue:>6,}  /  {extracted:,}  "
      f"({(extracted - empty_issue)    / extracted * 100:.1f}%)")
print(f"    ANALYSIS : {extracted - empty_analysis:>6,}  /  {extracted:,}  "
      f"({(extracted - empty_analysis) / extracted * 100:.1f}%)")

# ── Label distribution ────────────────────────────────────────────────────────
label_counts = Counter()
with open(OUTPUT_PATH) as f:
    for line in f:
        label_counts[json.loads(line)["label"]] += 1

print()
print(f"  Label distribution:")
for lbl, cnt in sorted(label_counts.items()):
    name = "ACCEPTED" if lbl == 1 else "REJECTED"
    print(f"    Label {lbl} ({name}) : {cnt:>6,}  ({cnt / extracted * 100:.1f}%)")

# ── Sample output (first 3 records) ──────────────────────────────────────────
print()
print("  Sample output (first 3 records):")
with open(OUTPUT_PATH) as f:
    for i, line in enumerate(f):
        if i >= 3:
            break
        rec = json.loads(line)
        print(f"\n  {'─' * 52}")
        print(f"  ID       : {rec['id']}  |  Label: {rec['label']}")
        print(f"  ISSUE    : {str(rec['ISSUE'])[:120]}"
              f"{'...' if len(str(rec['ISSUE']))    > 120 else ''}")
        print(f"  ANALYSIS : {str(rec['ANALYSIS'])[:120]}"
              f"{'...' if len(str(rec['ANALYSIS'])) > 120 else ''}")

print(f"\n  Output saved to → {OUTPUT_PATH}")

  ISSUE + ANALYSIS EXTRACTION COMPLETE
  Input          : cjpe_train_rr_segmented_CPU.jsonl
  Output         : cjpe_track_issue_analysis_extracted.jsonl
  Total records  : 32,229

  Field coverage (docs that HAVE the field):
    ISSUE    :  9,076  /  32,229  (28.2%)
    ANALYSIS : 28,432  /  32,229  (88.2%)

  Label distribution:
    Label 0 (REJECTED) : 18,873  (58.6%)
    Label 1 (ACCEPTED) : 13,356  (41.4%)

  Sample output (first 3 records):

  ────────────────────────────────────────────────────
  ID       : 2020_1  |  Label: 0
  ISSUE    : Uday Umesh Lalit, J. These appeals arise out of the Judgment and Order dated 09.12.2015 passed by the Division Bench of ...
  ANALYSIS : 

  ────────────────────────────────────────────────────
  ID       : 2020_2  |  Label: 0
  ISSUE    : Indira Banerjee, J. These appeals are against the judgment and order dated 21.11.2006 passed by the Madurai Bench of Mad...
  ANALYSIS : 

  ────────────────────────────────────────────────────
  ID       : 2